# Causal forest estimation

Run from the repository root, with the raw OULAD files in `raw/` (see README).

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
for p in [root, *root.parents]:
    if (p / "raw").is_dir():
        root = p
        break
BASE = str(root)

In [1]:
!pip install -q econml

import pandas as pd, numpy as np, joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from econml.dml import CausalForestDML, LinearDML
from scipy.stats import kruskal, pearsonr

SEED = 42
CANONICAL_ATE = 0.004786482048864249

full = pd.read_csv(f'{BASE}/oulad_full_corpus.csv')

confounders = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
               'disability', 'code_presentation', 'code_module',
               'num_of_prev_attempts', 'studied_credits']
categorical_cols = [c for c in confounders
                    if c not in ('num_of_prev_attempts', 'studied_credits')]

n_missing = int(full['imd_band'].isna().sum())
full['imd_band'] = full['imd_band'].fillna('Missing')
print(f'imd_band NaN -> "Missing": {n_missing}   (expect 830)')

X = full[confounders]
T = full['oucontent_clicks'].values
Y = full['performance_gain'].values
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print(f'confounder matrix: {X_encoded.shape}')
assert X_encoded.shape == (17529, 40), f'shape drift: {X_encoded.shape}'

cf = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=SEED),
                     model_t=HistGradientBoostingRegressor(random_state=SEED),
                     discrete_treatment=False, honest=True, min_samples_leaf=50,
                     n_estimators=500, cv=5, random_state=SEED)
cf.fit(Y, T, X=X_encoded)

cf_ate = float(cf.ate(X_encoded))
cf_ci = cf.ate_interval(X_encoded, alpha=0.05)
full['cate'] = cf.effect(X_encoded)
cf_ite = full['cate'].values

print(f'\nfresh fit ATE : {cf_ate!r}')
print(f'canonical constant   : {CANONICAL_ATE!r}')
print(f'difference           : {abs(cf_ate - CANONICAL_ATE):.3e}')

joblib.dump(cf, f'{BASE}/cf_canonical_model.pkl')
cf_saved = joblib.load(f'{BASE}/cf_canonical_model.pkl')
saved_ate = float(cf_saved.ate(X_encoded))
print(f'saved .pkl ATE       : {saved_ate!r}')
print(f'fresh vs saved       : {abs(cf_ate - saved_ate):.3e}')

assert abs(cf_ate - CANONICAL_ATE) < 1e-6, 'fit diverged from canonical'
assert abs(saved_ate - CANONICAL_ATE) < 1e-6, 'saved .pkl diverged from canonical'
print('canonical constant all agree to 1e-6.')

print(f'\nCATE mean = {full["cate"].mean()!r}')
print(f'     sd   = {full["cate"].std()!r}')
print(f'CF ATE 95% CI = [{float(cf_ci[0])!r}, {float(cf_ci[1])!r}]')

full[['id_student', 'code_module', 'code_presentation', 'cate']].to_csv(
    f'{BASE}/oulad_full_corpus_with_cate_v2.csv', index=False)
print('\nSaved oulad_full_corpus_with_cate_v2.csv')

In [2]:
r, p_val = pearsonr(T, Y)
print(f'Pearson (unconditional): r = {r!r}   p = {p_val!r}')

X_ols = pd.get_dummies(X, drop_first=True)
ols = LinearRegression().fit(np.column_stack([T, X_ols.values]), Y)
ols_ate = float(ols.coef_[0])
print(f'OLS coefficient on T   : {ols_ate!r}')

ldml = LinearDML(model_y=HistGradientBoostingRegressor(random_state=SEED),
                 model_t=HistGradientBoostingRegressor(random_state=SEED),
                 discrete_treatment=False, cv=5, random_state=SEED)
ldml.fit(Y, T, X=X_encoded)
ldml_ate = float(ldml.ate(X_encoded))
ldml_ci = ldml.ate_interval(X_encoded, alpha=0.05)
print(f'Linear DML             : {ldml_ate!r}')
print(f'                    CI : [{float(ldml_ci[0])!r}, {float(ldml_ci[1])!r}]')

cf_nh = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=SEED),
                        model_t=HistGradientBoostingRegressor(random_state=SEED),
                        discrete_treatment=False, honest=False, min_samples_leaf=50,
                        n_estimators=500, cv=5, random_state=SEED)
cf_nh.fit(Y, T, X=X_encoded)
nh_ate = float(cf_nh.ate(X_encoded))
nh_ci = cf_nh.ate_interval(X_encoded, alpha=0.05)

ablation = pd.DataFrame([
    {'config': 'OLS', 'ate': ols_ate, 'ci_lower': None, 'ci_upper': None},
    {'config': 'Linear DML', 'ate': ldml_ate,
     'ci_lower': float(ldml_ci[0]), 'ci_upper': float(ldml_ci[1])},
    {'config': 'CausalForestDML honest=False', 'ate': nh_ate,
     'ci_lower': float(nh_ci[0]), 'ci_upper': float(nh_ci[1])},
    {'config': 'CausalForestDML honest=True (PROPOSED)', 'ate': cf_ate,
     'ci_lower': float(cf_ci[0]), 'ci_upper': float(cf_ci[1])},
])
ablation['ci_width'] = ablation['ci_upper'] - ablation['ci_lower']
ablation['excludes_zero'] = np.where(
    ablation['ci_lower'].isna(), '-',
    np.where((ablation['ci_lower'] > 0) | (ablation['ci_upper'] < 0), 'yes', 'NO'))
print()
print(ablation.to_string(index=False, float_format=lambda v: f'{v:.9f}'))
ablation.to_csv(f'{BASE}/ablation_results_v2.csv', index=False)

print(f'\nhonest=False CI width / honest=True width = '
      f'{ablation.loc[2, "ci_width"] / ablation.loc[3, "ci_width"]:.2f}x')

Pearson (unconditional): r = np.float64(0.07942946799776814)   p = np.float64(6.143092240259981e-26)
OLS coefficient on T   : 0.0048976713680425115
Linear DML             : 0.003706448346069936
                    CI : [0.0027153375815471273, 0.004697559110592745]

                                config         ate     ci_lower    ci_upper    ci_width excludes_zero
                                   OLS 0.004897671          NaN         NaN         NaN             -
                            Linear DML 0.003706448  0.002715338 0.004697559 0.001982222           yes
          CausalForestDML honest=False 0.004567208 -0.001104927 0.010239343 0.011344270            NO
CausalForestDML honest=True (PROPOSED) 0.004786482  0.002576218 0.006996746 0.004420528           yes

honest=False CI width / honest=True width = 2.57x


In [3]:
print(full['oucontent_clicks'].describe().to_string())
print(f"\n5th/95th percentile: {full['oucontent_clicks'].quantile([0.05, 0.95]).values}")

sd_t = float(full['oucontent_clicks'].std())
per_sd = cf_ate * sd_t
print(f'\nSD of oucontent_clicks       : {sd_t!r}')
print(f'effect per 1 SD of engagement: {per_sd!r}')
print(f'effect per 100 clicks        : {cf_ate * 100!r}')

assert abs(sd_t - 781.7294105984906) < 1e-6, f'SD drift: {sd_t!r}'
assert abs(per_sd - 3.7417) < 1e-3, f'per-SD drift: {per_sd!r}'

count    17529.000000
mean       571.081066
std        781.729411
min          1.000000
25%         52.000000
50%        248.000000
75%        811.000000
max       9308.000000

5th/95th percentile: [2.000e+00 2.152e+03]

SD of oucontent_clicks       : 781.7294105984906
effect per 1 SD of engagement: 3.7417337908989055
effect per 100 clicks        : 0.47864820488642495



In [4]:
edu_order = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent',
             'HE Qualification', 'Post Graduate Qualification']
full['highest_education'] = pd.Categorical(full['highest_education'],
                                           categories=edu_order, ordered=True)
print(full['highest_education'].value_counts().reindex(edu_order).to_string())

edu_tab = (full.groupby('highest_education', observed=True)['cate']
           .agg(['mean', 'std', 'count']).reindex(edu_order))
print('\nMean CATE by highest_education (ascending attainment):')
print(edu_tab.to_string(float_format=lambda v: f'{v:.9f}'))

lta = edu_tab.loc['Lower Than A Level', 'mean']
pg = edu_tab.loc['Post Graduate Qualification', 'mean']
print(f'\nMARGINAL ratio (LTA / PostGrad) = {lta / pg!r}')

highest_education
No Formal quals                 121
Lower Than A Level             5897
A Level or Equivalent          8228
HE Qualification               3059
Post Graduate Qualification     224

Mean CATE by highest_education (ascending attainment):
                                   mean         std  count
highest_education                                         
No Formal quals             0.004757543 0.001993664    121
Lower Than A Level          0.005313276 0.001975399   5897
A Level or Equivalent       0.004791915 0.002095291   8228
HE Qualification            0.003894224 0.002112153   3059
Post Graduate Qualification 0.002919123 0.002482156    224

MARGINAL ratio (LTA / PostGrad) = np.float64(1.82016149798897)


In [5]:
groups = [full.loc[full['highest_education'] == lvl, 'cate'].values
          for lvl in edu_order]
stat, p = kruskal(*groups)
print(f'Kruskal-Wallis H = {stat!r}')
print(f'p                = {p!r}')

k = len(edu_order)
n = len(full)
epsilon_sq = (stat - k + 1) / (n - k)
print(f'\nk = {k}   n = {n:,}')
print(f'epsilon-squared  = {epsilon_sq!r}')



Kruskal-Wallis H = np.float64(1250.8225139518163)
p                = np.float64(1.5282757741308744e-269)

k = 5   n = 17,529
epsilon-squared  = np.float64(0.07114942444372382)




In [6]:
from sklearn.model_selection import LeaveOneGroupOut

modules = full['code_module'].values
logo = LeaveOneGroupOut()

logo_results = []
for train_idx, test_idx in logo.split(X_encoded, Y, groups=modules):
    held = full['code_module'].iloc[test_idx].unique()[0]

    col = f'code_module_{held}'
    dummy_sum = (int(X_encoded.iloc[train_idx][col].sum())
                 if col in X_encoded.columns else 'reference level (AAA)')

    cf_fold = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=SEED),
                              model_t=HistGradientBoostingRegressor(random_state=SEED),
                              discrete_treatment=False, honest=True, min_samples_leaf=50,
                              n_estimators=500, cv=5, random_state=SEED)
    cf_fold.fit(Y[train_idx], T[train_idx], X=X_encoded.iloc[train_idx])
    a_tr = float(cf_fold.ate(X_encoded.iloc[train_idx]))
    a_ho = float(cf_fold.ate(X_encoded.iloc[test_idx]))

    logo_results.append({'held_out_module': held,
                         'ate_train_modules': a_tr, 'ate_heldout_module': a_ho})
    print(f'Held out {held}: train = {a_tr!r}')
    print(f'{"":>13}held-out = {a_ho!r}')
    print(f'{"":>13}dummy sum in training = {dummy_sum}')

logo_df = pd.DataFrame(logo_results)
logo_df.to_csv(f'{BASE}/logo_cv_results_v2.csv', index=False)
print('\nSaved logo_cv_results_v2.csv (NOT logo_cv_results.csv — that name also')

print()
print(logo_df.to_string(index=False, float_format=lambda v: f'{v:.9f}'))
print(f"\nheld-out mean      = {logo_df['ate_heldout_module'].mean()!r}")
print(f"held-out sample SD = {logo_df['ate_heldout_module'].std(ddof=1)!r}")

insample = full.groupby('code_module', observed=True)['cate'].mean()
chk = logo_df.set_index('held_out_module').join(insample.rename('in_sample'))
from scipy.stats import spearmanr
rho, sp = spearmanr(chk['in_sample'], chk['ate_heldout_module'])
print(f'\nSpearman(in-sample, held-out) = {rho!r}   p = {sp!r}')

Held out AAA: train = 0.005180660834040104
             held-out = 0.005134498401981559
             dummy sum in training = reference level (AAA)
Held out BBB: train = 0.004601755040156671
             held-out = 0.004422901583499215
             dummy sum in training = 0
Held out CCC: train = 0.005704304592685034
             held-out = 0.0056248188114857624
             dummy sum in training = 0
Held out DDD: train = 0.004812546283587952
             held-out = 0.005111669120150022
             dummy sum in training = 0
Held out EEE: train = 0.0042564708556382365
             held-out = 0.003894902906543969
             dummy sum in training = 0
Held out FFF: train = 0.0025118340180876556
             held-out = 0.0022380621693435464
             dummy sum in training = 0

Saved logo_cv_results_v2.csv (NOT logo_cv_results.csv — that name also

held_out_module  ate_train_modules  ate_heldout_module
            AAA        0.005180661         0.005134498
            BBB        0.004601

In [7]:
print('=' * 78)
print('DDD DIAGNOSTIC')
print('=' * 78)

MODS = ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF']

def per_module(effects):
    s = pd.Series(effects, index=full.index)
    return s.groupby(full['code_module'], observed=True).mean().reindex(MODS)

base = pd.DataFrame({'cf': per_module(cf_ite), 'ldml': per_module(ldml.effect(X_encoded))})
base['abs_diff'] = (base['cf'] - base['ldml']).abs()
base['diff_vs_ATE'] = base['abs_diff'] / cf_ate
print(base.to_string(float_format=lambda v: f'{v:.6f}'))

print('\n--- 2. does DDD survive the leaf sweep? ---')
sweep = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=SEED),
                        model_t=HistGradientBoostingRegressor(random_state=SEED),
                        discrete_treatment=False, honest=True, min_samples_leaf=5,
                        n_estimators=500, cv=5, random_state=SEED)
sweep.fit(Y, T, X=X_encoded, cache_values=True)

leaf_rows = {}
for leaf in [5, 15, 25, 50, 100]:
    if leaf != 5:
        sweep.min_samples_leaf = leaf
        sweep.refit_final()
    leaf_rows[leaf] = per_module(sweep.effect(X_encoded))
leaf_tab = pd.DataFrame(leaf_rows)
print(leaf_tab.to_string(float_format=lambda v: f'{v:.6f}'))
ddd = leaf_tab.loc['DDD']
print(f'\nDDD across leaves: min {ddd.min()!r}  max {ddd.max()!r}')
print(f'  spread = {ddd.max() - ddd.min()!r}   sign changes = {bool((ddd > 0).nunique() > 1)}')

print('\n--- 3. does DDD survive across seeds? ---')
seed_rows = {}
for s in [42, 1, 7, 123]:
    cs = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=s),
                         model_t=HistGradientBoostingRegressor(random_state=s),
                         discrete_treatment=False, honest=True, min_samples_leaf=50,
                         n_estimators=500, cv=5, random_state=s)
    cs.fit(Y, T, X=X_encoded)
    seed_rows[s] = per_module(cs.effect(X_encoded))
    print(f'  seed {s:>4} done')
seed_tab = pd.DataFrame(seed_rows)
print()
print(seed_tab.to_string(float_format=lambda v: f'{v:.6f}'))
dds = seed_tab.loc['DDD']
print(f'\nDDD across seeds: mean {dds.mean()!r}  sd {dds.std(ddof=1)!r}')

w = pd.DataFrame({'module': full['code_module'],
                  'cf': cf_ite, 'ldml': ldml.effect(X_encoded)})
spread = w.groupby('module', observed=True).agg(
    cf_mean=('cf', 'mean'), cf_sd=('cf', 'std'),
    ldml_mean=('ldml', 'mean'), ldml_sd=('ldml', 'std'), n=('cf', 'size')).reindex(MODS)
print(spread.to_string(float_format=lambda v: f'{v:.6f}'))

leaf_tab.to_csv(f'{BASE}/ddd_leaf_sweep.csv')
seed_tab.to_csv(f'{BASE}/ddd_seed_sweep.csv')

print('\n' + '-' * 78)
print('-' * 78)
print('structure and the divergence is a specification difference worth reporting:')
print()
print()

DDD DIAGNOSTIC

                  cf      ldml  abs_diff  diff_vs_ATE
code_module                                          
AAA         0.004476  0.001573  0.002903     0.606573
BBB         0.005054  0.005559  0.000504     0.105378
CCC         0.000129 -0.000747  0.000876     0.183113
DDD         0.004694 -0.000848  0.005542     1.157813
EEE         0.005500  0.005866  0.000366     0.076382
FFF         0.006370  0.006737  0.000367     0.076636

--- 2. does DDD survive the leaf sweep? ---
                  5         15       25       50       100
code_module                                               
AAA          0.004181  0.004272 0.004367 0.004476 0.004523
BBB          0.005159  0.005033 0.005001 0.005054 0.005020
CCC         -0.000298 -0.000057 0.000040 0.000129 0.000157
DDD          0.004268  0.004463 0.004609 0.004694 0.004857
EEE          0.005601  0.005520 0.005528 0.005500 0.005488
FFF          0.006571  0.006461 0.006417 0.006370 0.006350

DDD across leaves: min 0.004267628